# Yomu masked-inpainting experiment — guarded harness

This notebook is synthetic-only by default. It does not install packages, download weights, mount Drive, or send Yomu/user images anywhere. Before adding a model adapter, verify the exact checkpoint license/revision and Colab unit balance. Use a full immutable 40-character source revision and a SHA-256 of the single weight file or canonical artifact manifest. If usage cannot be observed, run at most one image and stop for owner approval before continuing.

In [ ]:
from pathlib import Path
import json, os, platform, time, hashlib, re
from PIL import Image, ImageDraw
import numpy as np

OUT = Path('/content/yomu_colab_inpainting')
OUT.mkdir(parents=True, exist_ok=True)
MAX_IMAGES_PER_MODEL = 1  # increase only after observed unit usage + owner approval
OWNER_RECORDED_CU_BEFORE = None  # enter manually from Colab UI; never guessed
print({'python': platform.python_version(), 'platform': platform.platform(), 'output': str(OUT)})

In [ ]:
# Runtime probe. A paid plan does not guarantee any particular GPU. This does not inspect account billing or Colab compute-unit balance.
from datetime import datetime, timezone
print({'observed_at_utc': datetime.now(timezone.utc).isoformat()})
import subprocess
try:
    smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,driver_version', '--format=csv,noheader'], capture_output=True, text=True, timeout=10)
    print({'nvidia_smi_rc': smi.returncode, 'gpu': smi.stdout.strip() or 'NO_GPU', 'stderr': smi.stderr.strip()[:300]})
except Exception as exc:
    print({'gpu_probe': 'UNAVAILABLE', 'exception': type(exc).__name__})
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    vram = torch.cuda.get_device_properties(0).total_memory if gpu else None
    print({'torch': torch.__version__, 'cuda_runtime': torch.version.cuda, 'gpu_name': gpu, 'vram_bytes': vram})
except Exception as exc:
    print({'torch_probe': 'UNAVAILABLE', 'exception': type(exc).__name__})
try:
    import psutil
    disk = psutil.disk_usage(str(OUT))
    print({'host_ram_total_bytes': psutil.virtual_memory().total, 'host_ram_available_bytes': psutil.virtual_memory().available, 'disk_total_bytes': disk.total, 'disk_free_bytes': disk.free})
except Exception:
    print({'host_ram': 'NOT_AVAILABLE'})

In [ ]:
# Deterministic synthetic line-art / caption fixture, generated entirely in memory.
W, H = 640, 896
original = Image.new('RGB', (W, H), 'white')
draw = ImageDraw.Draw(original)
draw.rectangle((70, 70, 570, 820), outline=(30, 30, 30), width=4)
draw.ellipse((180, 160, 460, 440), outline=(25, 25, 25), width=6)
draw.arc((210, 200, 430, 420), 0, 180, fill=(60, 60, 60), width=4)
draw.line((130, 560, 510, 560), fill=(30, 30, 30), width=3)
draw.line((130, 590, 510, 650), fill=(70, 70, 70), width=2)
ground_truth = original.copy()
overlay_text = 'SYNTHETIC TEST'
text_origin = (198, 285)
text_bbox = draw.textbbox(text_origin, overlay_text)
draw.text(text_origin, overlay_text, fill=(0, 0, 0))
mask = Image.new('L', (W, H), 0)
padding = 8
ImageDraw.Draw(mask).rectangle((text_bbox[0]-padding, text_bbox[1]-padding, text_bbox[2]+padding, text_bbox[3]+padding), fill=255)
original.save(OUT / 'synthetic_source.png')
ground_truth.save(OUT / 'ground_truth_clean.png')
mask.save(OUT / 'synthetic_mask.png')
source_array, gt_array, mask_array = np.asarray(original), np.asarray(ground_truth), np.asarray(mask) > 0
overlay_outside_mask = int(np.any(source_array != gt_array, axis=2)[~mask_array].sum())
if overlay_outside_mask != 0:
    raise RuntimeError('synthetic_overlay_exceeds_authorized_mask')
display(original, mask)
print({'source_sha256': hashlib.sha256((OUT / 'synthetic_source.png').read_bytes()).hexdigest(), 'ground_truth_sha256': hashlib.sha256((OUT / 'ground_truth_clean.png').read_bytes()).hexdigest(), 'mask_sha256': hashlib.sha256((OUT / 'synthetic_mask.png').read_bytes()).hexdigest(), 'size': original.size, 'mask_pixels': int(mask_array.sum()), 'overlay_outside_mask_pixels': overlay_outside_mask})

## Candidate adapter boundary

No candidate is selected or downloaded by this notebook. After source/license approval and a pinned local model install, define `infer(image_rgb, mask_l)` to return one RGB PIL donor image with exactly the source dimensions. Record its model ID, immutable revision, weight SHA-256, code revision, precision, steps, and license before calling the harness. A prompt-only editor is not a mask-conditioned inpainting model.

In [ ]:
def safe_compose(source, donor, mask):
    if source.size != donor.size or source.size != mask.size:
        raise ValueError('shape_mismatch')
    final = Image.composite(donor.convert('RGB'), source.convert('RGB'), mask.convert('L'))
    a, b, m = np.asarray(source.convert('RGB')), np.asarray(final), np.asarray(mask) > 0
    outside_changed = int(np.any(a != b, axis=2)[~m].sum())
    return final, outside_changed

def score_inside_mask(candidate, ground_truth, mask):
    if candidate.size != ground_truth.size or candidate.size != mask.size:
        raise ValueError('ground_truth_shape_mismatch')
    c = np.asarray(candidate.convert('RGB'), dtype=np.float32)
    g = np.asarray(ground_truth.convert('RGB'), dtype=np.float32)
    m = np.asarray(mask.convert('L')) > 0
    err = c[m] - g[m]
    mae = float(np.mean(np.abs(err)))
    mse = float(np.mean(np.square(err)))
    return {'masked_mae': mae, 'masked_rmse': float(np.sqrt(mse)), 'masked_psnr_db': None if mse == 0 else float(20 * np.log10(255 / np.sqrt(mse))), 'perfect_masked_match': mse == 0}

def run_one_candidate(model_id, model_revision, weights_sha256, license_id, infer, model_load_seconds=None):
    if MAX_IMAGES_PER_MODEL != 1:
        raise RuntimeError('first_smoke_must_be_one_image')
    if not all([model_id, model_revision, weights_sha256, license_id]):
        raise ValueError('pin_model_revision_hash_and_license_first')
    if not re.fullmatch(r'[0-9a-fA-F]{40}', model_revision):
        raise ValueError('model_revision_must_be_full_immutable_commit')
    if not re.fullmatch(r'[0-9a-fA-F]{64}', weights_sha256):
        raise ValueError('weights_hash_must_be_sha256_or_manifest_sha256')
    if 'torch' not in globals() or not torch.cuda.is_available():
        raise RuntimeError('CUDA_GPU_NOT_AVAILABLE')
    torch.cuda.reset_peak_memory_stats()
    try:
        import psutil
        ram_before = psutil.virtual_memory().available
    except Exception:
        ram_before = None
    started = time.perf_counter()
    donor = infer(original.copy(), mask.copy())
    inference_seconds = time.perf_counter() - started
    if not isinstance(donor, Image.Image):
        raise TypeError('adapter_must_return_PIL_image')
    final, outside = safe_compose(original, donor, mask)
    gt_metrics = score_inside_mask(final, ground_truth, mask)
    safe_model_id = model_id.replace('/', '_')
    raw_path = OUT / f'{safe_model_id}_raw_donor.png'
    final_path = OUT / f'{safe_model_id}_safe_final.png'
    donor.convert('RGB').save(raw_path)
    final.save(final_path)
    try:
        ram_after = psutil.virtual_memory().available
    except Exception:
        ram_after = None
    report = {'compositor_integrity': 'PASS' if outside == 0 else 'REJECT', 'quality_status': 'REVIEW_REQUIRED', 'text_removal_verified': 'NOT_AUTOMATED', 'human_visual_review': 'REQUIRED', 'ground_truth_metrics_inside_mask': gt_metrics, 'model_id': model_id, 'revision': model_revision, 'weights_sha256': weights_sha256, 'license': license_id, 'image_size': original.size, 'model_load_seconds': model_load_seconds, 'inference_seconds': inference_seconds, 'host_ram_available_before_bytes': ram_before, 'host_ram_available_after_bytes': ram_after, 'peak_allocated_vram_bytes': torch.cuda.max_memory_allocated(), 'peak_reserved_vram_bytes': torch.cuda.max_memory_reserved(), 'outside_mask_changed_pixels': outside, 'source_sha256': hashlib.sha256((OUT / 'synthetic_source.png').read_bytes()).hexdigest(), 'ground_truth_sha256': hashlib.sha256((OUT / 'ground_truth_clean.png').read_bytes()).hexdigest(), 'mask_sha256': hashlib.sha256((OUT / 'synthetic_mask.png').read_bytes()).hexdigest(), 'raw_donor_sha256': hashlib.sha256(raw_path.read_bytes()).hexdigest(), 'final_sha256': hashlib.sha256(final_path.read_bytes()).hexdigest(), 'owner_recorded_CU_before': OWNER_RECORDED_CU_BEFORE, 'owner_recorded_CU_after': None}
    (OUT / f'{safe_model_id}_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    display(original, donor, final)
    return report

# Intentionally not invoked: no approved model adapter, weights, GPU, or CU visibility in this task.

## Stop conditions

Stop if the assigned GPU is unavailable, unit use cannot be observed, a weight/license cannot be verified, dimensions differ, an adapter touches pixels outside the mask, or one run exceeds the approved budget. Do not retry an ambiguous provider execution automatically. Human visual review is required for text removal, line-art continuity, texture, faces/anatomy, and collateral edits; exact outside-mask equality is a hard gate, not a quality score. This notebook must not disconnect or shut down the runtime; leave it connected as requested by the owner.